# Rain Alert Model Comparison

So sanh cac model cho bai toan phan loai co mua / khong mua sau khi dung du doan luong mua voi threshold `>= 1.0 mm`.

## 1. Setup

In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == 'Results' else Path.cwd()
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from src.aq_course_ml.config import RAIN_ALERT_THRESHOLD_MM

RESULT_DIR = PROJECT_DIR / 'Results'
COMPARISON_CSV_PATH = RESULT_DIR / 'rain_alert_model_comparison.csv'

plt.style.use('seaborn-v0_8-whitegrid')

## 2. Load Comparison Metrics

In [ ]:
comparison_df = pd.read_csv(COMPARISON_CSV_PATH)
comparison_df = comparison_df.sort_values('f1_rain', ascending=False).reset_index(drop=True)

print(f'Threshold: predicted rain >= {RAIN_ALERT_THRESHOLD_MM:g} mm')
comparison_df

## 3. Compare Each Rain Alert Metric

Cac metric chinh cho class `rain`:

- `accuracy`: ty le du doan dung tren toan bo test set.
- `precision_rain`: khi model bao co mua, ty le thuc su co mua.
- `recall_rain`: trong cac gio thuc su co mua, ty le model bat duoc.
- `f1_rain`: can bang giua precision va recall cho class co mua.
- `false_alarm_rate`: ty le bao mua nham trong cac truong hop khong mua.
- `miss_rate`: ty le bo sot mua trong cac truong hop thuc su co mua.

In [ ]:
def plot_metric(metric, title, ylabel='Score', higher_is_better=True):
    ordered = comparison_df.sort_values(metric, ascending=not higher_is_better)
    plt.figure(figsize=(11, 4.5))
    ax = sns.barplot(data=ordered, x='model', y=metric, hue='model', palette='viridis', legend=False)
    for container in ax.containers:
        ax.bar_label(container, fmt='%.3f', padding=3, fontsize=8)
    plt.title(title)
    plt.xlabel('Model')
    plt.ylabel(ylabel)
    plt.ylim(0, 1.08)
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()

In [ ]:
plot_metric('accuracy', 'Accuracy by Model')

In [ ]:
plot_metric('precision_rain', 'Precision for Rain Class by Model')

In [ ]:
plot_metric('recall_rain', 'Recall for Rain Class by Model')

In [ ]:
plot_metric('f1_rain', 'F1 Score for Rain Class by Model')

In [ ]:
plot_metric('false_alarm_rate', 'False Alarm Rate by Model', ylabel='Error rate', higher_is_better=False)

In [ ]:
plot_metric('miss_rate', 'Miss Rate by Model', ylabel='Error rate', higher_is_better=False)

## 4. Metric Summary Table

In [ ]:
summary_columns = [
    'model',
    'accuracy',
    'precision_rain',
    'recall_rain',
    'f1_rain',
    'false_alarm_rate',
    'miss_rate',
    'tn',
    'fp',
    'fn',
    'tp',
]

comparison_df[summary_columns].round(4)

## 5. Confusion Matrix for Each Model

Hang la nhan that, cot la nhan du doan. Label `0` la khong mua, label `1` la co mua.

In [ ]:
def plot_confusion_matrix(row, ax):
    matrix = [[row['tn'], row['fp']], [row['fn'], row['tp']]]
    sns.heatmap(
        matrix,
        annot=True,
        fmt='.0f',
        cmap='Blues',
        cbar=False,
        xticklabels=['Pred: no rain', 'Pred: rain'],
        yticklabels=['Actual: no rain', 'Actual: rain'],
        ax=ax,
    )
    ax.set_title(row['model'])
    ax.set_xlabel('Predicted label')
    ax.set_ylabel('Actual label')


n_models = len(comparison_df)
n_cols = 3
n_rows = -(-n_models // n_cols)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4.5 * n_rows))
axes = axes.flatten()

for ax, (_, row) in zip(axes, comparison_df.iterrows()):
    plot_confusion_matrix(row, ax)

for ax in axes[n_models:]:
    ax.axis('off')

plt.suptitle(f'Confusion Matrices by Model (threshold >= {RAIN_ALERT_THRESHOLD_MM:g} mm)', y=1.02, fontsize=14)
plt.tight_layout()

## 6. Read the Trade-off

- Model co `recall_rain` cao se bat duoc nhieu gio co mua hon, nhung co the tang bao dong gia.
- Model co `precision_rain` cao se bao mua chac hon, nhung co the bo sot mot so gio mua.
- `f1_rain` la metric can bang hop ly neu can vua bat duoc mua, vua giam bao dong gia.
- `miss_rate` thap quan trong neu muc tieu la khong bo sot mua.
- `false_alarm_rate` thap quan trong neu muc tieu la tranh bao mua nham.